# Async programming

Asynchronous programming is a programming paradigm that allows us to write code that can run concurrently without blocking the main thread.\
It is particularly useful for I/O-bound tasks, such as network requests or file operations,\
where the program can continue executing other tasks while waiting for the I/O operation to complete.\

First what happens when we execute a synchronous code?\
When we execute a synchronous code, the program will block and wait for each instruction to complete before moving on to the next one.\
For example, if we have a function that takes a long time to execute, the program will block and wait for that function to complete before moving on to the next instruction:
```python
import time

def slow_task() -> str:
    print("Starting slow task")
    time.sleep(3)
    print("Slow task completed")
    return "slow"

result = slow_task()
print(f"Result: {result}")
```
This can lead to a poor user experience, as the program may become unresponsive while waiting for the long-running function to complete.

On the other hand, with asynchronous programming, we can write code that can run concurrently without blocking the main thread.\
This allows us to write more efficient and responsive programs, as we can continue executing other tasks while waiting for the long-running function to complete.\
We will start with a simple example of an asynchronous function that simulates a long-running task:
```python
import time

async def slow_task() -> str:
    print("Starting slow task")
    time.sleep(5)
    print("Slow task completed")
    return "slow"

result = slow_task()
print(f"Result: {result}")  # Output: Result: <coroutine object slow_task at 0x00000224E7073040>
```
This code only return a Coroutine object, and it does not execute the function.\
Because coroutines are not executed until they are awaited, we need to use the `await` keyword to execute the function and wait for it to complete\
We will see how to use `await` in next sections,\
but for now, we can see that the function is not executed and the program continues to execute the next instruction without waiting for the slow task to complete.

In Python, we can use the `asyncio` library to write asynchronous code.\
This library provides a set of tools to manage asynchronous tasks, such as creating and running coroutines, managing event loops, and handling exceptions.\
In this notebook, we will explore the basics of asynchronous programming in Python using the `asyncio` library.

## Asyncio

First of all, we need to understand the difference between asynchronous and synchronous programming.\
A synchronous code is one where each instruction waits for the previous one to execute, while asynchronous code does not wait for deferred instructions and continues with its execution.\
It is basically what is defined as parallel programming.\
Parallelism consists of performing multiple operations at the same time.

### Basics Methods

In this section, we will explore the basic methods provided by the `asyncio` library to manage asynchronous tasks.\
We will cover the following methods:
- `asyncio.run()`
- `await`
- `asyncio.gather()`

#### Run
This is the main method to execute asynchronous functions.\
In Python scripts we must use `.run()` to create an event loop and execute the main function\
Remember tu use `async def` to create asynchronous functions:
```python
import asyncio

async def main() -> str:
    print("Slow task completed")
    return "slow"

if __name__ == "__main__":
    asyncio.run(main())
```

Remember in `Jupyter Notebook`, we **MUST** directly use `await` without creating an event loop (with .run()), as Jupyter already has an event loop running in the background.\
It will be explained in the next section

#### Await

Await is important in asynchronous programming as it helps us wait for a result and prevents the function from continuing to execute until the called method or coroutine is complete.\
To call other functions within another function, it is `INDISPENSABLE` to use await to call and wait for the function to finish executing.
```python
import asyncio

async def main() -> str:
    print("Slow task started")
    await asyncio.sleep(3)  # <- This is where we wait for the task to complete
    return "slow"

if __name__ == "__main__":
    asyncio.run(main())
```

In Jupyter Notebook we can simply do:

In [7]:
import asyncio


async def slow_task() -> str:
    print("Slow task started")
    await asyncio.sleep(3)
    return "slow"


if __name__ == "__main__":
    await slow_task()

Slow task started


'slow'

#### Gather

`asyncio.gather()` is a method that allows us to execute multiple coroutines concurrently and wait for all of them to complete.\
It takes multiple coroutines as arguments and returns a list of their results in the same order as they were passed.\
If any of the coroutines raises an exception, `asyncio.gather()` will raise that exception and cancel the remaining coroutines.\
However, if we set `return_exceptions=True`, it will return the exceptions as part of the results instead of raising them.

Again in Python scripts we must use `.run()` to create an event loop and execute the main function:
```python
import asyncio


async def task1() -> str:
    await asyncio.sleep(1)
    return "result1"


async def task2() -> None:
    await asyncio.sleep(2)
    msg = "An error occurred in task2"
    raise ValueError(msg)


async def task3() -> str:
    await asyncio.sleep(3)
    return "result3"


async def main() -> None:
    try:
        results = await asyncio.gather(
            task3(), task2(), task1(), return_exceptions=True
        )
        print(results)
    except Exception as e:
        print(f"Exception: {e}")


if __name__ == "__main__":
    asyncio.run(main())
```

While in Jupyter Notebook we can directly use `await` without creating an event loop:

In [3]:
import asyncio
from typing import Literal, NoReturn


async def task1() -> Literal["result1"]:
    print("Task 1 started")
    await asyncio.sleep(1)
    return "result1"


async def task2() -> NoReturn:
    print("Task 2 started")
    await asyncio.sleep(2)
    msg = "An error occurred in task2"
    raise ValueError(msg)


async def task3() -> Literal["result3"]:
    print("Task 3 started")
    await asyncio.sleep(3)
    return "result3"


results = await asyncio.gather(task3(), task2(), task1(), return_exceptions=True)
print(results)

Task 3 started
Task 2 started
Task 1 started
['result3', ValueError('An error occurred in task2'), 'result1']


### Tasks

`Tasks` allow us to wrap coroutines in a way that they can be scheduled to run concurrently.\
A `Task` is a subclass of `Future` that represents an asynchronous operation that can be scheduled to run concurrently with other tasks.\
When we create a task, it is automatically scheduled to run on the event loop, and we can use the `await` keyword to wait for its result.

#### Create tasks

To create a task, we can use `asyncio.create_task()`:
```python
import asyncio

loop = asyncio.get_event_loop()
task_2 = loop.create_task(quick_task())
```

In Jupyter Notebook, we can directly use `asyncio.create_task()` without creating an event loop:
```jupyter
import asyncio

task_2 = asyncio.create_task(quick_task())
await task_2
```

In [6]:
import asyncio


async def slow_task() -> str:
    await asyncio.sleep(3)
    print("Slow task completed")
    return "slow"


async def quick_task() -> str:
    print("Quick task completed")
    return "quick"


# Create tasks
task1 = asyncio.create_task(slow_task())
task2 = asyncio.create_task(quick_task())

# Await tasks
await asyncio.gather(task1, task2)

Quick task completed
Slow task completed


['slow', 'quick']

#### Delete tasks

To delete a task, we can use the `cancel()` method.\
This will cancel the task and it will not be executed.\
If the task is already running, it will raise a `CancelledError` exception.
```python
import asyncio


async def slow_task() -> str:
    print("Slow task started")
    await asyncio.sleep(3)
    return "slow"


async def main() -> None:
    task1 = asyncio.create_task(slow_task())
    await asyncio.sleep(1)
    task1.cancel()
    await task1


try:
    asyncio.run(main())
except asyncio.CancelledError:
    print("Task was cancelled")
```

In [4]:
import asyncio


async def slow_task() -> str:
    print("Slow task started")
    await asyncio.sleep(3)
    return "slow"


async def main() -> None:
    task1 = asyncio.create_task(slow_task())
    await asyncio.sleep(1)
    task1.cancel()
    await task1


try:
    await main()
except asyncio.CancelledError:
    print("Exception Raised: Task was cancelled")

{<Task pending name='Task-47' coro=<_async_in_context.<locals>.run_in_context() running at C:\Mine\Programming\Proyects\Scraping\Scraping-Libraries-in-Python\.venv\Lib\site-packages\ipykernel\utils.py:60> wait_for=<Task pending name='Task-48' coro=<Kernel.shell_main() running at C:\Mine\Programming\Proyects\Scraping\Scraping-Libraries-in-Python\.venv\Lib\site-packages\ipykernel\kernelbase.py:621> wait_for=<Task pending name='Task-49' coro=<InteractiveShell.run_cell_async() running at C:\Mine\Programming\Proyects\Scraping\Scraping-Libraries-in-Python\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3400> cb=[IPythonKernel._cancel_on_sigint.<locals>.cancel_unless_done()() at C:\Mine\Programming\Proyects\Scraping\Scraping-Libraries-in-Python\.venv\Lib\site-packages\ipykernel\ipkernel.py:336, Task.task_wakeup()]> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Mine\Programming\Proyects\Scraping\Scraping-Libraries-in-Python\.venv\Lib\site-packages\zm

#### Task Group

The `asyncio.gather` is a powerful method to execute multiple coroutines concurrently, but it does not provide a way to manage the tasks it creates.\
`asyncio.TaskGroup` is a new feature in Python 3.11 that allows us to manage a group of tasks and handle their exceptions in a more structured way.\
It provides a context manager that allows us to create a group of tasks and wait for them to complete.\
If any of the tasks raises an exception, the context manager will automatically cancel the remaining tasks and raise the exception.
Example in Python script:
```python
import asyncio


async def task1() -> Literal["result1"]:
    await asyncio.sleep(1)
    return "result1"


async def task2() -> NoReturn:
    print("Task 2 started")
    await asyncio.sleep(2)
    msg = "An error occurred in task2"
    raise ValueError(msg)


async def task3() -> Literal["result3"]:
    print("Task 3 started")
    await asyncio.sleep(3)
    return "result3"


async def main() -> None:
    tasks = []
    try:
        async with asyncio.TaskGroup() as tg:
            t1 = tg.create_task(task3())
            t2 = tg.create_task(task2())
            t3 = tg.create_task(task1())
            tasks.append([t1, t2, t3])
        all_results = [t.result() for t in tasks]
        print(f"All results: {all_results}")
    except* Exception as e:
        print(f"Exception: {e.exceptions}")


asyncio.run(main())
```

In [6]:
import asyncio


async def task1() -> Literal["result1"]:
    await asyncio.sleep(1)
    return "result1"


async def task2() -> NoReturn:
    print("Task 2 started")
    await asyncio.sleep(2)
    msg = "An error occurred in task2"
    raise ValueError(msg)


async def task3() -> Literal["result3"]:
    print("Task 3 started")
    await asyncio.sleep(3)
    return "result3"


tasks = []
try:
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(task3())
        t2 = tg.create_task(task2())
        t3 = tg.create_task(task1())
        tasks.append([t1, t2, t3])

    all_results = [t.result() for t in tasks]
    print(f"All results: {all_results}")
except* Exception as e:
    print(f"Exception: {e.exceptions}")

Task 3 started
Task 2 started
Exception: (ValueError('An error occurred in task2'),)


### Synchronization

In asynchronous programming, we often need to synchronize access to shared resources to prevent race conditions and ensure data integrity.\
Python's `asyncio` library provides several synchronization primitives, such as `Lock`, `Event`, `Condition`, and `Semaphore`, to help us manage concurrent access to shared resources.\
In this section, we will explore these synchronization primitives and how to use them in asynchronous programming.

#### Lock

A `Lock` is a synchronization primitive that allows only one task to access a shared resource at a time.\
When a task acquires a lock, it prevents other tasks from acquiring the same lock until it is released.\
This is useful for protecting critical sections of code that access shared resources, such as a shared variable or a file.
```python
import asyncio

lock = asyncio.Lock()


async def worker(lock: asyncio.Lock, worker_id: int) -> None:
    print(f"Worker {worker_id} is waiting to acquire the lock")
    async with lock:
        print(f"Worker {worker_id} has acquired the lock")
        await asyncio.sleep(3)
        print(f"Worker {worker_id} is releasing the lock")


async def main() -> None:
    await asyncio.gather(worker(lock, 1), worker(lock, 2), worker(lock, 3))

asyncio.run(main())
```


In [10]:
import asyncio

lock = asyncio.Lock()


async def worker(lock_parm: asyncio.Lock, worker_id: int) -> None:
    # Emojins used for better visualization of the lock status
    print(f"Worker {worker_id} is waiting ⏳ to acquire the lock")
    async with lock_parm:
        print(f"Worker {worker_id} has acquired 🤩 the lock")
        await asyncio.sleep(3)
        print(f"Worker {worker_id} has released ✅ the lock")


await asyncio.gather(worker(lock, 1), worker(lock, 2), worker(lock, 3))

Worker 1 is waiting ⏳ to acquire the lock
Worker 1 has acquired 🤩 the lock
Worker 2 is waiting ⏳ to acquire the lock
Worker 3 is waiting ⏳ to acquire the lock
Worker 1 has release ✅ the lock
Worker 2 has acquired 🤩 the lock
Worker 2 has release ✅ the lock
Worker 3 has acquired 🤩 the lock
Worker 3 has release ✅ the lock


[None, None, None]

#### Semaphore